# Data Preparation

Este notebook é dedicado para a preparação dos nossos dados para análise. Aqui iremos passar pelos seguintes processos:

- Definição dos tipos de cada coluna;

- Transformação do formato _wide_ para _long_;

- Carregamento dos dados em um arquivo _.parquet_, assim mantendo salvo os tipos de variável de nossas colunas, reduzindo o espaço utilizado no disco rígido, facilitando a leitura seletiva de colunas e aumentando a velocidade no processamento dos dados durante as análises.

Nosso dataset segue o formato _wide_ para alocação dos dados de emissão de cada ano. Iremos transformar para o formato _long_, assim criando as colunas "Ano" e "CO2e (t)" para fazer a alocação dos valores.
Esse processo irá facilitar no momento geração dos gráficos e sumarização dos nossos dados.

_Importando bibliotecas e definindo o caminho de acesso dos dados:_

In [ ]:
import os
import pandas as pd
import pyarrow

caminho = "data/raw"

### Definindo as funções para aplicar nos Dataset's:

- **read_and_structure_file()**: 
  - Responsável pela leitura da tabela, remoção das colunas que não serão de interesse e inserção da coluna com a informação da Unidade Federativa da linha.

- **reduction_memory_usage()**:
  - Responsável por alterar os tipos de dado nas colunas determinadas como categóricas, assim reduzindo consideravelmente o uso de memória no processamentos dos dados durante o uso do dataframe.

- **wide_to_long()**: 
  - Responsável por transformar nossos dados de emissão do formato wide para o formato long, gerando as colunas _Ano_ e _CO2e (t)_. Na função, determinamos o tipo de variável de ambas as colunas como numéricas (_Ano_ como números inteiros e _CO2e (t)_ como números com casas decimais).

In [ ]:
def read_and_structure_file(uf_name:str):

    dataframe = pd.read_csv(f"{caminho}/{uf_name}/ar6.csv", low_memory=False, sep=",")            # Lendo arquivo

    dataframe.drop(columns=['Recorte', 'Bioma', 'Emissão/Remoção/Bunker', 'Gás'], inplace=True)   # Removendo colunas que não são de interesse para essa análise
    
    dataframe.insert(loc=7, column="UF", value=uf_name)                                           # Inserindo coluna com a Unidade Federativa

    return dataframe

def reduction_memory_usage(dataframe:pd.DataFrame):

    categorical = dataframe.columns.tolist()[0:8]

    for column in categorical:
        dataframe[column] = dataframe[column].astype('category')

    return dataframe

def wide_to_long(dataframe:pd.DataFrame):

    dataframe = (dataframe.set_index(['Setor de emissão',
                                      'Categoria emissora',
                                      'Sub-categoria emissora',
                                      'Produto ou sistema',
                                      'Detalhamento',
                                      'Atividade geral',
                                      'Cidade',
                                      'UF'])
                          .stack()
                          .reset_index()
                          .rename(columns={"level_8" : "Ano", 0 : "CO2e (t)"}))

    dataframe['CO2e (t)'] = dataframe['CO2e (t)'].astype(float)
    dataframe['Ano'] = dataframe['Ano'].astype(int)

    return dataframe

Gerandos os dataframe's dos estados da região sul:

In [ ]:
files_name = os.listdir(caminho)

for uf in files_name:

    if uf == 'SC':

        df_sc = wide_to_long(reduction_memory_usage(read_and_structure_file(uf)))

    elif uf == 'PR':

        df_pr = wide_to_long(reduction_memory_usage(read_and_structure_file(uf)))

    elif uf == 'RS':

        df_rs = wide_to_long(reduction_memory_usage(read_and_structure_file(uf)))

    else:
        continue

In [ ]:
df_sc.info(memory_usage='deep')

df_pr.info(memory_usage='deep')

df_rs.info(memory_usage='deep')

### Gerando Dataset da Região Sul:

Na célula abaixo iremos concatenar os dataframe´s de Santa Catarina, Rio Grande do Sul e Paraná, para enfim gerar o dataset que será utilizado para análise. Para isso, precisamos primeiro padronizar as categorias das colunas categóricas, para que não haja perda no tipo de variável no momento da concatenação.

Padronizamos e concatenamos da seguinte forma:

In [ ]:
categorical_cols = ['Setor de emissão',
                    'Categoria emissora',
                    'Sub-categoria emissora',
                    'Produto ou sistema',
                    'Detalhamento',
                    'Atividade geral',
                    'Cidade',
                    'UF']

for column in categorical_cols:

    categories = pd.concat([
        df_pr[column],
        df_sc[column],
        df_rs[column]
    ]).dropna().unique()

    df_pr[column] = df_pr[column].cat.set_categories(categories)
    df_sc[column] = df_sc[column].cat.set_categories(categories)
    df_rs[column] = df_rs[column].cat.set_categories(categories)

df_sul = pd.concat(
    [df_pr, df_sc, df_rs],
    ignore_index=True)

### Salvando o Dataframe em formato Parquet:

In [ ]:
df_sul.to_parquet(
    "data/processed/Sul.parquet",
    engine="pyarrow",
    index=False)

_Confirmando se os dados foram salvos corretamente:_

In [ ]:
df_teste = pd.read_parquet("data/processed/Sul.parquet", engine="pyarrow")

print("Dataframe tratado no notebook:")
df_sul.info(memory_usage="deep")

print("\nDataframe gerado pelo arquivo .parquet lido após os tratamento:")
df_teste.info(memory_usage="deep")